# VISUALIZE AUGMENTATION OUTPUT

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import io
from PIL import Image
from typing import Any

from emma import AugmentedDataset, EmmaDataset
from emma.augment import HDF5Source
import hdf5plugin

import matplotlib.pyplot as plt

## > DATASET

In [ ]:
DATASET_ROOT = Path("PATH/TO/EMMa")
AUGMENTATION_PATH = Path("PATH/TO/THRESHOLDING-H5-FILE")
SOURCE_NAME = "thresholding_masks"

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Please set DATASET_ROOT to a valid EMMa dataset directory. Got: {DATASET_ROOT}"
    )

if not AUGMENTATION_PATH.is_file():
    raise FileNotFoundError(
        f"Please set AUGMENTATION_PATH to a valid augmentation file. Got: {AUGMENTATION_PATH}"
    )

In [ ]:
ds = EmmaDataset(DATASET_ROOT)

In [ ]:
source = HDF5Source(SOURCE_NAME, AUGMENTATION_PATH, mode="r")

In [ ]:
ads = AugmentedDataset(ds).register(source)

## > VISUALIZE

In [ ]:
NUM_ITEMS = 10

augmented_asins = list(ads.source(SOURCE_NAME).keys())
selected_asins = augmented_asins[:NUM_ITEMS]

In [ ]:
def first_image_rgb(ds: EmmaDataset, rec: dict[str, Any]) -> Image.Image:
    image_dict = ds.get_images(rec)
    _, first_img_bytes = next(iter(image_dict.items()))
    return Image.open(io.BytesIO(first_img_bytes)).convert("RGB")


def mask_to_pil(mask: np.ndarray) -> Image.Image:
    mask = np.asarray(mask).squeeze()
    mask_vis = (mask > 0).astype(np.uint8) * 255
    return Image.fromarray(mask_vis, mode="L")


def overlay_mask(image: Image.Image, mask: np.ndarray) -> np.ndarray:
    image_np = np.asarray(image).astype(np.float32) / 255.0
    mask_bool = np.asarray(mask).squeeze().astype(bool)
    if mask_bool.shape != image_np.shape[:2]:
        raise ValueError(
            f"Mask shape {mask_bool.shape} does not match image shape {image_np.shape[:2]}."
        )

    overlay = image_np.copy()
    overlay[mask_bool] = 0.55 * overlay[mask_bool] + 0.45 * np.array([1.0, 0.0, 0.0])
    return np.clip(overlay, 0.0, 1.0)


def show_segmentation(ads: AugmentedDataset, asin: str) -> None:
    rec = ads.get_record(asin)
    
    image = first_image_rgb(ads, rec)
    mask = ads.get_augmentation(SOURCE_NAME, asin)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"{asin} | image={image.size[::-1]} mask={mask.shape}")

    axes[0].imshow(image)
    axes[0].set_title("Original")

    axes[1].imshow(mask_to_pil(mask), cmap="gray", vmin=0, vmax=255)
    axes[1].set_title("Saved mask")

    if mask.squeeze().shape == np.asarray(image).shape[:2]:
        axes[2].imshow(overlay_mask(image, mask))
        axes[2].set_title("Overlay")
    else:
        axes[2].text(
            0.5,
            0.5,
            "Mask and image\nshapes differ",
            ha="center",
            va="center",
        )
        axes[2].set_title("Overlay")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
for asin in selected_asins:
    show_segmentation(ads, asin)